# Lab 0: See It First, AI Vision Demo

**Duration:** 1 hour (5 min briefing, 25 min setup, 20 min demo, 10 min review). **Level:** absolute beginner.

| Student name | Station no. | Date |
|---|---|---|
| | | |

(Double-click this cell to type your name.)

Run each code cell with **Shift + Enter**. The commands you see are the real commands. Lines that start with `!` or `%` run on the Jetson, and their output appears under the cell while they run.

## 1. Overview
Before you learn how it works, see what it can do. In this lab your Jetson starts an AI vision program. It watches the camera and names the objects it sees, live, on the GPU. Then you play a game against the clock.

You do not need to understand anything yet. In Labs 1 to 9 you learn every part of this, and in Lab 9 you build a project of your own.

**By the end of this lab you will be able to:**
- Start the demo from a notebook cell.
- Describe what the AI program shows on the screen.
- Say what the Jetson does that an ordinary computer would find hard to do this fast.
- Write down three things you want to learn in this course.

## 2. Part 0: One-Time Setup (20 to 40 min, with your instructor)

Your Jetson comes with JetPack 6.2 only. This part installs the Python tools used in Lab 0 and Labs 4 to 9. It is done **once for the whole Jetson**, not once for each student. It downloads several gigabytes, so it takes time. Watch the output: you see every package arrive.

> **If the check below says everything is installed**, someone has done this part already. Skip to **Part A**.

> **Warning.** If a step stops with a message about PyTorch or the GPU, do not try to fix it yourself. Do **not** run a plain `pip install torch`. Show the message to your instructor.

### Step 1: Take Your Turn and Check What Is Already Installed
Only **one student at a time** installs on a Jetson, because two installs would break each other. This cell puts you in the queue. If a classmate is installing, it tells you who and waits, and when it is your turn it checks again, so you may find that everything is already done. When you pass this cell, the check below is up to date.

In [ ]:
import labview
labview.setup_begin()

In [ ]:
import importlib.util, subprocess, sys

def gpu_ok():
    # True only if PyTorch imports AND a real calculation runs on the GPU
    code = ("import torch,sys; a=torch.ones(64,64,device='cuda'); "
            "sys.exit(0 if torch.cuda.is_available() and float((a@a).sum())==262144.0 else 1)")
    return subprocess.run([sys.executable, "-c", code], capture_output=True).returncode == 0

for name in ["cv2", "cupy", "ultralytics", "onnx", "ipywidgets", "openpyxl"]:
    print(f"{name:<12}", "installed" if importlib.util.find_spec(name) else "MISSING")
missing = [n for n in ["cv2", "cupy", "ultralytics", "onnx", "ipywidgets", "openpyxl"] if not importlib.util.find_spec(n)]
ready = gpu_ok()
print(f"{'PyTorch GPU':<12}", "works" if ready else "NOT READY")
if not missing and ready:
    labview.setup_end()
    print("Everything is installed. Skip to Part A.")

Every line that says `MISSING` or `NOT READY` is installed by one of the steps below. If nothing is missing, skip to Part A. If you do install, keep going to the end: the last step lets the next student in.

### Step 2: Allow `sudo`
A notebook has no terminal where `sudo` can ask for your password, so this cell asks for it once. Type your password in the box that appears (nothing is shown). It is kept only inside this notebook until you restart the kernel.

In [ ]:
import labview
labview.sudo_login()

### Step 3: Install the System Packages
`apt` installs software for the whole Ubuntu system. `-y` means "answer yes to the questions". This installs OpenCV, the camera tools and the programs used in Labs 2 and 3.

In [ ]:
!sudo apt update

In [ ]:
!sudo apt install -y python3-venv python3-pip python3-opencv v4l-utils nano tree htop git

### Step 4: Install jtop
jtop is the monitor you use in Lab 1. It goes into the system Python, outside the course environment, so the command uses `/usr/bin/pip3`.

In [ ]:
!sudo -H /usr/bin/pip3 install -U jetson-stats

### Step 5: Install CuPy
CuPy is NumPy for the GPU (Lab 5). CuPy 14 needs NumPy 2, but the Jetson libraries need NumPy 1, so we ask for CuPy 13. `%pip` installs into the course environment that this notebook uses.

In [ ]:
%pip install "cupy-cuda12x<14" "numpy<2"

### Step 6: Install PyTorch with GPU Support
Only run the install cell if Step 1 said `PyTorch GPU  NOT READY`. This is a large download from the Jetson AI Lab index, which has the build made for JetPack 6.2.

In [ ]:
%pip install torch torchvision --index-url https://pypi.jetson-ai-lab.io/jp6/cu126 --force-reinstall
%pip install "numpy<2"

Newer PyTorch builds need the **cuDSS** library, which plain JetPack does not include. This cell checks for it and installs it into the course environment if it is missing.

In [ ]:
import glob, os, subprocess, sys

ACTIVATE = os.path.expanduser("~/venvs/jetson/bin/activate")
VENV = os.path.expanduser("~/venvs/jetson")

def torch_error():
    return subprocess.run([sys.executable, "-c", "import torch"], capture_output=True, text=True).stderr

if "libcudss" in torch_error():
    print("PyTorch needs the cuDSS library. Installing it into the course environment.")
    subprocess.run([sys.executable, "-m", "pip", "install", "nvidia-cudss-cu12==0.7.1.6", "--no-deps"], check=True)
    libs = glob.glob(VENV + "/**/libcudss.so.0", recursive=True)
    folder = os.path.dirname(libs[0])
    line = f'export LD_LIBRARY_PATH="{folder}:$LD_LIBRARY_PATH"  # CUDSS_DIR (added by the Lab 0 notebook)\n'
    if "CUDSS_DIR" not in open(ACTIVATE).read():
        open(ACTIVATE, "a").write(line)          # every new "lab start" finds it
    os.environ["LD_LIBRARY_PATH"] = folder + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("PyTorch GPU:", "works" if gpu_ok() else "NOT READY. Show this notebook to your instructor.")

### Step 7: Install Ultralytics
Ultralytics provides the YOLO26 AI models (Labs 7 to 9). The `-U` flag upgrades it, because YOLO26 needs a recent version.

Ultralytics brings its own OpenCV, but the course uses the OpenCV that comes with JetPack. So we remove the pip one afterwards, then pin the versions that fit NumPy 1.

In [ ]:
%pip install -U ultralytics

In [ ]:
%pip uninstall -y opencv-python opencv-python-headless
%pip install "numpy<2" "cupy-cuda12x<14" "scipy>=1.11,<1.15" "onnx>=1.12.0,<2.0.0" "onnxslim>=0.1.82"

> **Note.** While this runs you may see a red `ERROR` about `opencv-python` and a line that says `GPU device discovery failed`. Both are harmless. Keep waiting until the cell finishes.

### Step 8: Restart the Kernel
PyTorch and NumPy were replaced while this notebook was open. In the menu choose **Kernel, Restart Kernel**, then run only the cells below. You do not need `sudo` again.

### Step 9: Download the AI Model Files
The models are downloaded once into `~/students/_shared/models`, and every student's folder reaches them through the link called `models`. The labs then need no internet. The two sample pictures that come with Ultralytics are copied beside them.

In [ ]:
import glob, os, shutil

MODELS = os.path.expanduser("~/students/_shared/models")
os.makedirs(MODELS, exist_ok=True)
here = os.getcwd()
os.chdir(MODELS)
from ultralytics import YOLO
for name in ("yolo26n-cls.pt", "yolo26n.pt", "yolo26s.pt"):
    YOLO(name)
import ultralytics
assets = os.path.join(os.path.dirname(ultralytics.__file__), "assets")
for picture in glob.glob(assets + "/*.jpg"):
    shutil.copy(picture, ".")
os.chdir(here)
print("Model files:", sorted(os.path.basename(f) for f in glob.glob(MODELS + "/*")))

### Step 10: Check the Whole Setup
This runs the check program of the course. Expect: NumPy 1.x, CUDA `True`, and no pip `opencv-python`.

In [ ]:
import pathlib, labview
REPO = pathlib.Path(labview.__file__).resolve().parent.parent
!python {REPO}/verify_setup.py

Part 0 is done. Let the next student in:

In [ ]:
labview.setup_end()

**Your answers:** (double-click, type after each colon)

- Did every line say installed or works?
- Which step took the longest?

## 3. Part A: Start the Demo (5 min)

Run the cell below. Wait about 20 seconds while the AI model loads. The live picture from your camera then appears **inside this notebook**, with buttons above it.

If you see `Cannot open the camera`, check that the USB camera is plugged in, and that you started with `lab start yourname camera 0` (the number must be one that `lab cameras` shows).

In [ ]:
import sys, pathlib, labview
sys.path.append(str(pathlib.Path(labview.__file__).resolve().parent / "lab0_demo"))
import demo
view = demo.notebook()

## 4. Part B: Explore (5 min)

Point the camera at yourself, your neighbour and the things on your desk. Coloured boxes appear around the objects the AI recognises, with a name and a number such as `0.87`. The number is how sure the AI is, where `1.00` means completely sure.

| What you see | Meaning |
|---|---|
| Coloured box with a name | The AI found that kind of object here |
| Number such as 0.87 | Confidence. 0.87 means 87% sure. |
| 28 FPS (bottom right) | Frames per second. How many pictures the Jetson understands every second. |

Write the answers in the Results table in Section 6.

## 5. Part C: The 60 Second Challenge (10 min)

Now the game. Find as many different kinds of object as you can in 60 seconds. Show each object to the camera. Each new kind counts once, so three cups count as one.

1. Press **Start / Restart** above the picture to start the clock.
2. Show objects to the camera one at a time. Names appear on the right as you find them.
3. Move the **Confidence** slider any time. A higher limit means the AI must be more sure before it names something.
4. When time is up, the picture shows your total and your list. Press **Start / Restart** to try again.
5. Press **Stop** when you finish. This frees the camera.

> **Note.** The AI knows 80 kinds of everyday objects, such as person, cup, bottle, laptop, book, chair, cell phone and keyboard. If it does not name something, the object is probably not one of the 80.

Play at least two rounds. Try a different confidence limit in each round, for example 0.40 and 0.70, and see which kinds of objects disappear when the limit is high.

## 6. Results
Double-click this cell and type your answers after each colon.

- Number of FPS shown on your screen:
- Highest confidence number you saw:
- Challenge round 1, kinds found:
- Challenge round 2, kinds found:
- An object the AI named wrongly or missed:

## 7. Check Task
Show your instructor:

1. Start the demo yourself from the notebook.
2. Run one full 60 second challenge.
3. Point out a box, a name and a confidence number on the screen.
4. Close the demo with the **Stop** button.

## 8. Reflection
Answer in your own words in this cell. There are no wrong answers.

1. What surprised you most?
2. Why do you think the AI sometimes names an object wrongly?
3. Where could a camera that understands what it sees be useful in real life?
4. Write three things you hope to learn in this course.

## What Happens Next

| Lab | What you will learn |
|---|---|
| 1 to 3 | Your Jetson and the Linux commands you used today |
| 4 to 6 | Python, the GPU and the camera, so you can write programs like this |
| 7 to 8 | How the AI recognises objects, and how to make it much faster |
| 9 | Build your own AI vision project |

## 9. Troubleshooting

| Problem | Likely cause | What to do |
|---|---|---|
| `Cannot open the camera` | USB camera not plugged in, or you did not choose one | Plug it in firmly. In the SSH terminal, type `lab cameras`, then start again with `lab start yourname camera 0`. |
| Password not accepted | Typing error. Nothing shows while you type. | Run the `sudo_login` cell again. |
| `No module named ...` | Part 0 has not been done on this Jetson | Run Part 0 from Step 1, or tell the instructor. |
| The picture is very slow | Another student is using the GPU, or the plain model is used | Tell the instructor. |
| The picture does not appear | The widgets did not load | Reload the page (F5) and run the cell again. |
| Nothing is recognised | Too dark, or the object is too far | Add light and move the object closer. |